In [3]:
import mlflow
import os
import sys
from dotenv import load_dotenv
import pandas as pd
import lightgbm
import time
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
import json

In [4]:
pd.set_option("display.max_columns", 100)
load_dotenv()
data_path = os.getenv("DATA_PATH")
src_path = os.getenv("SRC_PATH")
sys.path.append(src_path)
json_path = os.path.join(data_path, "processed/split_info.json")
with open(json_path) as f:
    json_info = json.load(f)
train_end = json_info.get("train_end")
val_end = json_info.get("validation_end")
from about_data.data_load import load_df
from about_data.split import temporal_split
from features.engineering import add_all_features
from model.preprocessor_pipe_evalueate import create_pipeline, evaluate_model, get_preprocessor

full_df = load_df(data_path)
train, val, test = temporal_split(full_df, train_end, val_end)

In [5]:
map_dfs = {"train": train, "val": val}

x_dfs = {}
y_dfs = {}

for name, sample_df in map_dfs.items():
    x_dfs[name] = add_all_features(sample_df)
    y_dfs[name] = sample_df['isFraud']

In [6]:
y_dfs['train'].shape, x_dfs['train'].shape, y_dfs['val'].shape, x_dfs['val'].shape  

((413378,), (413378, 48), (88581,), (88581, 48))

In [8]:
model = lightgbm.LGBMClassifier(n_estimators=300, learning_rate=0.05, objective="binary", metric='average_precision', random_state=42, n_jobs=-1)

X_train = x_dfs['train']
X_val = x_dfs['val']
y_train = y_dfs['train']
y_val = y_dfs['val']

pipe = create_pipeline(model, get_preprocessor(X_train))

In [18]:
param_grid = {
    "model__n_estimators": [300, 500, 700, 1000],
    "model__learning_rate": [0.02, 0.03, 0.05, 0.08, 0.1],
    "model__num_leaves": [15, 31, 63, 127],
    "model__max_depth": [-1, 5, 7, 10, 15],
    "model__min_child_samples": [20, 50, 100, 200],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__reg_alpha": [0, 0.1, 0.5, 1.0],
    "model__reg_lambda": [0, 0.1, 0.5, 1.0, 5.0]
}

tscv = TimeSeriesSplit(n_splits=5)

In [19]:
grid = RandomizedSearchCV(
    pipe,
    param_distributions= param_grid,
    scoring="average_precision",
    cv = tscv,
    n_iter = 50,
    n_jobs = -1,
    random_state= 42,
    verbose=1,
    refit=True)

In [20]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("fraud-detection-final-tuning")

with mlflow.start_run(run_name="LightGBM_RandomizedSearch"):
    
    start = time.time()
    
    grid.fit(X_train, y_train)

    training_time = time.time() - start
    
    metrics = evaluate_model(grid, X_val, y_val)

    best_model = grid.best_estimator_

    mlflow.log_params({
        "model": "lightgbm",
        "search_method": "RandomizedSearchCV",
        "n_iter": 50,
        "cv": "TimeSeriesSplit",
        "n_splits": 5,
        "scoring": "average_precision",
        "feature_count": X_train.shape[1],
        "n_train": len(X_train),
        "n_validation": len(X_val)
    })

    mlflow.log_metric("best_cv_pr_auc", grid.best_score_)

    mlflow.log_metrics(metrics)

    mlflow.log_metric("training_time_seconds", training_time)

    mlflow.log_params({f"best_{key}": value for key, value in grid.best_params_.items()})


Fitting 5 folds for each of 50 candidates, totalling 250 fits
[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.078011 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13884
[LightGBM] [Info] Number of data points in the train set: 413378, number of used features: 5458
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311794
[LightGBM] [Info] Start training from score -3.311794
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
